## 1. Definición Formal del Problema

La detección de objetos se enfoca en una tarea más específica y exigente que la clasificación de imágenes o la segmentación semántica. En lugar de asignar una etiqueta global o clasificar píxeles sueltos, el objetivo es **localizar todas las instancias de una categoría de objeto** mediante cuadros delimitadores (*bounding boxes*).

En pocas palabras, el modelo no solo responde a la pregunta "¿está el objeto en la imagen?", sino que también debe resolver obligatoriamente "¿dónde está?".

### 1.1. Formulación Matemática

Formalmente, la función de nuestro modelo de detección (generalmente una red neuronal profunda parametrizada por los pesos $\theta$) se define como:

$$f_\theta: \mathbb{R}^{H \times W \times 3} \rightarrow \mathcal{B}$$

Donde:
* $H, W$: Son el alto y ancho de la imagen de entrada a color (3 canales).
* $\mathcal{B} = \{ (b_i, s_i) \}_{i=1}^N$: Es el conjunto de $N$ detecciones predichas por la red.
* $b_i = (x, y, w, h)$: Es el cuadro delimitador que encierra al objeto, definido por sus coordenadas, ancho y alto.
* $s_i$: Es la puntuación de confianza (*confidence score*) que indica la certeza del modelo de que la caja $b_i$ realmente contiene un objeto.

### 1.2. Desafíos Principales en la Detección

Para lograr que el modelo pase de los píxeles de entrada a estas cajas delimitadoras, la arquitectura y el entrenamiento deben superar tres obstáculos fundamentales:

* **Variación de Escala:** Un mismo objeto (como un vehículo o una persona) puede abarcar desde unos pocos píxeles hasta ocupar casi todo el encuadre. El modelo debe ser capaz de "ver" y procesar características a múltiples resoluciones.
* **Oclusión:** En entornos reales, los objetos suelen estar parcialmente tapados. La red debe aprender a inferir la caja delimitadora completa $b_i$ apoyándose en partes visibles limitadas.
* **Desbalance Extremo de Clases (Fondo vs. Objeto):** Si analizamos todas las posibles ubicaciones en una imagen, la inmensa mayoría corresponden simplemente a "fondo" sin interés. Este desbalance causa que, durante el entrenamiento, los aciertos en el fondo abrumen a los errores en los objetos reales, requiriendo el uso de funciones de pérdida especializadas \cite{lin2017focal}.

## 2. Arquitecturas Representativas en Detección de Objetos

Históricamente, los modelos de detección basados en *Deep Learning* se han dividido en dos grandes familias: los detectores de dos etapas (*two-stage*) y los de una etapa (*one-stage*). Cada familia representa un compromiso diferente entre la precisión espacial y el costo computacional.

### 2.1. Faster R-CNN (Two-Stage Detector)
Propuesto por Ren et al. \cite{ren2015faster}, es el arquetipo de los detectores de dos etapas y se ha mantenido como el estándar de oro para aplicaciones que priorizan la precisión. Funciona en dos fases claramente diferenciadas:

1. **Region Proposal Network (RPN):** Una sub-red convolucional ligera que se desliza sobre los mapas de características. Su función exclusiva es proponer "Regiones de Interés" (RoIs): cajas delimitadoras candidatas que tienen una alta probabilidad de contener *algún* objeto, discriminando únicamente entre *foreground* y *background*.
2. **Detector Head:** Para cada RoI propuesta, se extrae un parche de características de tamaño fijo (mediante *RoI Align*). Posteriormente, una red densa clasifica la región específica y aplica una regresión final para afinar las coordenadas de la caja.

**Innovación clave:** La RPN permitió que la red aprendiera a proponer regiones internamente, posibilitando el entrenamiento *end-to-end* y superando cuellos de botella algorítmicos.

### 2.2. YOLO (You Only Look Once - One-Stage Detector)
Introducido por Redmon et al. \cite{redmon2016you}, YOLO revolucionó el campo al plantear la detección como un único problema de regresión directa.

1. La imagen de entrada se divide en una cuadrícula (*grid*) de $S \times S$ celdas.
2. Cada celda de la parrilla es responsable de predecir simultáneamente $B$ cuadros delimitadores, sus puntuaciones de confianza y las probabilidades condicionales de clase.

**Innovación clave:** Al procesar la imagen completa en una única pasada, alcanza velocidades en tiempo real e incorpora un contexto global, lo que reduce drásticamente los falsos positivos en el fondo.

### 2.3. SSD (Single Shot MultiBox Detector - One-Stage Detector)
Desarrollado por Liu et al. \cite{liu2016ssd}, SSD ofrece un compromiso estratégico entre la extrema velocidad de YOLO y la precisión analítica de Faster R-CNN.

* Aplica cabezales de detección sobre **múltiples mapas de características a diferentes escalas**.
* Los mapas de capas más profundas (baja resolución) detectan objetos grandes, mientras que los mapas de capas más superficiales (alta resolución) detectan objetos pequeños.

#### Resumen Arquitectónico

| Arquitectura | Paradigma | Innovación Principal | Fortalezas / Debilidades |
| :--- | :--- | :--- | :--- |
| **Faster R-CNN** | Dos Etapas | Region Proposal Network (RPN) | **+** Muy alta precisión<br>**-** Lento computacionalmente |
| **YOLO** | Una Etapa | Detección unificada basada en grid | **+** Extremadamente rápido<br>**-** Menos preciso en objetos pequeños |
| **SSD** | Una Etapa | Detección multiescala con *anchor boxes* | **+** Buen balance velocidad-precisión<br>**-** Complejo de configurar |

## 3. Anatomía General de un Detector Moderno

Independientemente del paradigma al que pertenezcan, las arquitecturas modernas han convergido en un diseño modular estandarizado que consta de tres componentes principales interconectados:

### 3.1. Backbone (Extractor de Características)
Es la red convolucional profunda (ej. ResNet, MobileNet, VGG) que actúa como un *encoder*. Toma la matriz de píxeles original y genera una jerarquía de mapas de características semánticamente ricos a diferentes escalas de resolución.

### 3.2. Neck (Cuello - Fusión de Características)
Un módulo estructural (siendo **FPN - Feature Pyramid Network** \cite{lin2017feature} el más destacado) que combina los mapas de características de diferentes niveles del *backbone*. Su objetivo es enriquecer las representaciones fusionando la **información semántica robusta** de las capas profundas con la **información espacial de alta resolución** de las capas superficiales, facilitando la detección multiescala.

### 3.3. Head (Cabezal de Predicción)
Toma los mapas de características optimizados por el *Neck* y ejecuta las predicciones finales. Estructuralmente se bifurca en dos ramas paralelas:
* **Rama de Clasificación:** Predice la confianza ($s_i$) de que un cuadro contenga el objeto y a qué clase pertenece.
* **Rama de Regresión:** Ajusta las coordenadas continuas del cuadro delimitador ($b_i$) para que se ciña con la mayor exactitud geométrica posible a la instancia real.

## 4. Optimización: Funciones de Pérdida

Entrenar un detector de objetos es complejo porque la red debe aprender dos tareas de naturaleza distinta al mismo tiempo. Por ello, la pérdida total del modelo es siempre una suma ponderada:

$$ \mathcal{L}_{total} = \mathcal{L}_{cls} + \lambda \cdot \mathcal{L}_{loc} $$

Donde $\mathcal{L}_{cls}$ penaliza los errores de clasificación, $\mathcal{L}_{loc}$ penaliza los errores geométricos de la caja delimitadora, y $\lambda$ es un hiperparámetro que equilibra ambas tareas.

### 4.1. Pérdida de Clasificación ($\mathcal{L}_{cls}$)

El enfoque tradicional es usar la **Binary Cross-Entropy (BCE)**. Sin embargo, en detección enfrentamos un problema severo: en una imagen hay miles de cajas candidatas que son solo "fondo", y muy pocas que son objetos reales.

Para solucionar este desbalance, se utiliza la **Focal Loss** \cite{lin2017focal}. Esta función modifica la entropía cruzada añadiendo un factor de modulación que reduce el peso de los ejemplos "fáciles" (como el fondo claro) y obliga a la red a concentrarse en los objetos difíciles:

$$ \mathcal{L}_{FL}(p_t) = - \alpha_t (1 - p_t)^\gamma \log(p_t) $$

Cuando el modelo está seguro de una predicción ($p_t \rightarrow 1$), el término $(1 - p_t)^\gamma$ se acerca a cero, anulando casi por completo la penalización de ese ejemplo.

### 4.2. Pérdida de Localización ($\mathcal{L}_{loc}$)

Históricamente se utilizó la **Smooth L1 Loss**, ya que es menos sensible a valores atípicos (*outliers*) que una pérdida cuadrática estándar:

$$ \text{Smooth}_{L1}(x) = \begin{cases} 0.5 x^2 & \text{si } |x| < 1 \\ |x| - 0.5 & \text{en otro caso} \end{cases} $$

Sin embargo, la Smooth L1 optimiza cada coordenada (x, y, w, h) de forma independiente, lo cual no siempre se correlaciona con la métrica visual real de solapamiento.

Por ello, los modelos modernos utilizan **pérdidas basadas en IoU**. La más completa es la **CIoU Loss (Complete IoU)** \cite{zheng2020distance}, que penaliza tres aspectos geométricos simultáneamente:
1. La falta de solapamiento entre cajas.
2. La distancia normalizada entre los centros de las cajas ($\frac{\rho^2}{c^2}$).
3. La inconsistencia en la relación de aspecto ($\alpha v$).

$$ \mathcal{L}_{CIoU} = 1 - \text{IoU} + \frac{\rho^2(b_p, b_{gt})}{c^2} + \alpha v $$

## 5. Métricas de Evaluación en Detección

En clasificación, medir la "exactitud" (*accuracy*) es sencillo. En detección, necesitamos saber si el modelo predijo la clase correcta *y además* si la ubicó correctamente en el espacio.

### 5.1. Intersection over Union (IoU)
Es la métrica base. Mide geométricamente el solapamiento entre la caja predicha ($B_p$) y la caja real de *Ground Truth* ($B_{gt}$):

$$ \text{IoU} = \frac{\text{Area}(B_p \cap B_{gt})}{\text{Area}(B_p \cup B_{gt})} $$

### 5.2. Definiendo Aciertos y Errores
Establecemos un **umbral de IoU** (generalmente 0.5) para decidir si una detección es válida:
* **True Positive (TP):** La red predijo la clase correcta y el solapamiento con la caja real es mayor al umbral (ej. IoU > 0.5).
* **False Positive (FP):** La red detectó un objeto donde no lo hay, o el solapamiento es menor al umbral.
* **False Negative (FN):** Había un objeto real en la imagen, pero la red no lo detectó.

### 5.3. Precision, Recall y mAP
A partir de los TP, FP y FN, calculamos dos métricas fundamentales:
* **Precision:** De todas las cajas que la red dibujó, ¿qué porcentaje era realmente un objeto? ($\frac{TP}{TP + FP}$).
* **Recall:** De todos los objetos reales en la imagen, ¿qué porcentaje logró encontrar la red? ($\frac{TP}{TP + FN}$).

Como todo modelo arroja una "puntuación de confianza" por caja, podemos variar el umbral de aceptación para obtener múltiples pares de Precision y Recall. Esto forma una curva. El área bajo esta curva se conoce como **Average Precision (AP)**:

$$ \text{AP} = \int_0^1 p(r) \, dr $$

Finalmente, el **Mean Average Precision (mAP)** es simplemente el promedio del AP calculado para todas las clases del dataset. En competencias estrictas como MS COCO, se evalúa el **mAP@[0.5:0.95]**, que exige que el modelo sea preciso bajo múltiples niveles de exigencia de solapamiento.

## 6. Configuración del Entorno y Reproducibilidad

Comenzamos importando las librerías fundamentales y fijando las semillas de aleatoriedad. En investigación y docencia, garantizar que los experimentos sean reproducibles es innegociable.

In [ ]:
import os
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms as T
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models.detection.ssdlite import SSDLiteHead
from torchvision.ops import box_iou

# 1. Fijar semillas para reproducibilidad
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# 2. Configurar dispositivo (GPU si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de cómputo configurado: {device}")

Dispositivo de cómputo configurado: cuda


## 7. Adquisición y Preprocesamiento del Dataset (PennFudanPed)

Para garantizar agilidad en la clase, utilizaremos **PennFudanPedO128**, un subconjunto ligero de 170 imágenes del dataset PennFudanPed. 

**Importante (Conversión Geométrica):** Las anotaciones en PennFudanPed suelen venir en formato normalizado `[x_centro, y_centro, ancho, alto]`. Sin embargo, las arquitecturas de PyTorch (`torchvision`) exigen estrictamente coordenadas absolutas de las esquinas: `[x_min, y_min, x_max, y_max]`. Nuestro `Dataset` personalizado debe realizar esta transformación matemática al vuelo.

Un aspecto crítico en detección es la función de colación (`collate_fn`). Por defecto, PyTorch intenta apilar los tensores en una matriz regular (ej. `[batch_size, num_objetos, 4]`). Dado que una imagen puede tener 2 objetos y otra 10, esto arrojaría un error. 

Para solucionarlo, reescribimos la función para que empaquete las muestras en tuplas, respetando la dimensionalidad variable de cada elemento del lote.

In [ ]:
def download_pennfudan(root="data"):
    """Descarga PennFudanPed si no existe."""
    import urllib.request, zipfile
    url = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"
    zip_path = os.path.join(root, "PennFudanPed.zip")
    dataset_path = os.path.join(root, "PennFudanPed")
    if not os.path.exists(dataset_path):
        os.makedirs(root, exist_ok=True)
        print("Descargando PennFudanPed...")
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(root)
        os.remove(zip_path)
        print("Dataset listo.")
    return dataset_path


class PennFudanDataset(Dataset):
    """
    PennFudanPed → 2 clases: 0=background, 1=person
    Cada imagen tiene una máscara PNG donde cada píxel != 0 es una persona distinta.
    """
    def __init__(self, root, transforms=None):
        self.root = root
        self.transforms = transforms
        self.imgs  = sorted(os.listdir(os.path.join(root, "PNGImages")))
        self.masks = sorted(os.listdir(os.path.join(root, "PedMasks")))

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path  = os.path.join(self.root, "PNGImages", self.imgs[idx])
        mask_path = os.path.join(self.root, "PedMasks",  self.masks[idx])

        img  = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        # Cada valor único != 0 en la máscara es un peatón
        mask = np.array(mask)
        obj_ids = np.unique(mask)[1:]  # quita el 0 (fondo)

        boxes, labels = [], []
        for obj_id in obj_ids:
            pos = np.where(mask == obj_id)
            xmin, xmax = pos[1].min(), pos[1].max()
            ymin, ymax = pos[0].min(), pos[0].max()
            if xmax > xmin and ymax > ymin:          # ignora cajas degeneradas
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(1)                     # 1 = person

        boxes  = torch.tensor(boxes,  dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes":  boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target


def get_transform():
    return T.Compose([T.ToTensor()])


def collate_fn(batch):
    return tuple(zip(*batch))

## 8. Definición del Modelo Base (SSDLite con MobileNetV3)

Para esta implementación, optamos por **SSDLite** respaldado por un *backbone* **MobileNetV3**.

Entrenar un detector desde cero con pesos inicializados aleatoriamente requiere cientos de horas en clústers de GPUs para estabilizarse. Al utilizar un modelo preentrenado (*Transfer Learning*), aprovechamos la profunda jerarquía visual que el *backbone* ya extrajo de millones de imágenes. Esto nos permite enfocar nuestro esfuerzo computacional (y cognitivo) exclusivamente en el *Head* del modelo, logrando convergencia rápida y validando de manera práctica los conceptos aprendidos.

In [ ]:
def get_model(num_classes=2):
    """
    Carga SSDLite320 preentrenado en COCO y reemplaza la cabeza
    para num_classes clases.
    Compatible con distintas versiones de torchvision.
    """
    model = ssdlite320_mobilenet_v3_large(weights="DEFAULT")

    # Extrae in_channels recorriendo cada bloque del classification_head.
    # En versiones recientes cada bloque es un Sequential, no un Conv directo,
    # así que buscamos el primer Conv2d dentro de cada bloque.
    def _get_in_channels(sequential_block):
        for m in sequential_block.modules():
            if isinstance(m, torch.nn.Conv2d):
                return m.in_channels
        raise ValueError("No se encontró Conv2d en el bloque de la cabeza.")

    in_channels = [
        _get_in_channels(block)
        for block in model.head.classification_head.module_list
    ]
    num_anchors = model.anchor_generator.num_anchors_per_location()

    model.head = SSDLiteHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes,
        norm_layer=torch.nn.BatchNorm2d,
    )
    return model



Arquitectura SSDLite instanciada y enviada al dispositivo correctamente.


## 9. Implementación Manual de Métricas de Evaluación

La base de cualquier evaluación en detección es calcular el solapamiento entre la predicción y el *Ground Truth*.

In [ ]:
def _match_boxes(pred_boxes, gt_boxes, iou_thr):
    """Matching greedy: una GT solo puede matchear una predicción."""
    if len(pred_boxes) == 0 and len(gt_boxes) == 0:
        return 0, 0, 0
    if len(pred_boxes) == 0:
        return 0, 0, len(gt_boxes)
    if len(gt_boxes) == 0:
        return 0, len(pred_boxes), 0

    iou_mat   = box_iou(pred_boxes, gt_boxes)   # [P, G]
    matched_gt = set()
    tp = fp = 0

    for p_idx in range(len(pred_boxes)):
        best_iou, best_g = iou_mat[p_idx].max(0)
        if best_iou >= iou_thr and best_g.item() not in matched_gt:
            tp += 1
            matched_gt.add(best_g.item())
        else:
            fp += 1

    fn = len(gt_boxes) - len(matched_gt)
    return tp, fp, fn


def _compute_ap(pred_boxes, pred_scores, gt_boxes, iou_thr):
    """AP de una sola imagen con interpolación de 11 puntos."""
    if len(gt_boxes) == 0:
        return 1.0 if len(pred_boxes) == 0 else 0.0
    if len(pred_boxes) == 0:
        return 0.0

    # Ordena por score descendente
    order      = pred_scores.argsort(descending=True)
    pred_boxes = pred_boxes[order]

    iou_mat   = box_iou(pred_boxes, gt_boxes)
    matched_gt = set()
    tps, fps  = [], []

    for p_idx in range(len(pred_boxes)):
        best_iou, best_g = iou_mat[p_idx].max(0)
        if best_iou >= iou_thr and best_g.item() not in matched_gt:
            tps.append(1); fps.append(0)
            matched_gt.add(best_g.item())
        else:
            tps.append(0); fps.append(1)

    tps_cum = np.cumsum(tps)
    fps_cum = np.cumsum(fps)
    recalls    = tps_cum / len(gt_boxes)
    precisions = tps_cum / (tps_cum + fps_cum + 1e-6)

    # 11-point interpolation
    ap = 0.0
    for thr in np.linspace(0, 1, 11):
        prec_at_rec = precisions[recalls >= thr]
        ap += prec_at_rec.max() if len(prec_at_rec) > 0 else 0.0
    return ap / 11.0

## 10. Pipeline de Entrenamiento (Training Loop)

Entrenar un detector requiere manejar un diccionario de pérdidas. A diferencia de la clasificación simple, PyTorch nos devolverá un sumatorio de las pérdidas de regresión y clasificación de la arquitectura.

In [ ]:
def train_one_epoch(model, optimizer, loader, device):
    model.train()
    total_loss = 0.0
    for imgs, targets in loader:
        imgs    = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        losses = sum(loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, device, iou_threshold=0.5, score_threshold=0.5):
    """
    Calcula Precision, Recall y mAP@50 sobre el loader dado.

    Estrategia simple y directa:
      - Filtra predicciones por score_threshold
      - Hace matching greedy por IoU >= iou_threshold
      - Acumula TP/FP/FN globales → Precision y Recall
      - Calcula AP por imagen con 11-point interpolation → promedia (mAP)
    """
    model.eval()
    all_tp, all_fp, all_fn = 0, 0, 0
    all_aps = []

    for imgs, targets in loader:
        imgs = [img.to(device) for img in imgs]
        preds = model(imgs)

        for pred, target in zip(preds, targets):
            gt_boxes  = target["boxes"].to(device)
            pr_boxes  = pred["boxes"]
            pr_scores = pred["scores"]
            pr_labels = pred["labels"]

            # Filtra por score y clase 1 (person)
            mask = (pr_scores >= score_threshold) & (pr_labels == 1)
            pr_boxes  = pr_boxes[mask]
            pr_scores = pr_scores[mask]

            tp, fp, fn = _match_boxes(pr_boxes, gt_boxes, iou_threshold)
            all_tp += tp
            all_fp += fp
            all_fn += fn

            ap = _compute_ap(pr_boxes, pr_scores, gt_boxes, iou_threshold)
            all_aps.append(ap)

    precision = all_tp / (all_tp + all_fp + 1e-6)
    recall    = all_tp / (all_tp + all_fn + 1e-6)
    map50     = float(np.mean(all_aps)) if all_aps else 0.0

    return {"precision": precision, "recall": recall, "map50": map50}


def train(epochs=10, batch_size=4, lr=0.005, data_root="data"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Usando dispositivo: {device}")

    # Dataset
    dataset_path = download_pennfudan(data_root)
    full_dataset = PennFudanDataset(dataset_path, transforms=get_transform())

    n_val   = max(1, int(0.2 * len(full_dataset)))
    n_train = len(full_dataset) - n_val
    train_ds, val_ds = random_split(full_dataset, [n_train, n_val])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn, num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=1,          shuffle=False,
                              collate_fn=collate_fn, num_workers=0)

    # Modelo
    model = get_model(num_classes=2).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr,
                                momentum=0.9, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    # Loop
    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, optimizer, train_loader, device)
        scheduler.step()

        metrics = evaluate(model, val_loader, device)
        print(f"Época {epoch:02d}/{epochs} | "
              f"Loss: {loss:.4f} | "
              f"Precision: {metrics['precision']:.3f} | "
              f"Recall: {metrics['recall']:.3f} | "
              f"mAP@50: {metrics['map50']:.3f}")

    torch.save(model.state_dict(), "ssdlite_pennfudan.pth")
    print("\nModelo guardado en ssdlite_pennfudan.pth")
    return model, val_ds, device

In [ ]:
model, val_dataset, device = train(epochs=10, batch_size=4, lr=0.005)

## 11. Análisis y Visualización de Resultados

Finalmente, cargamos los pesos del mejor modelo y evaluamos su capacidad de generalización sobre el conjunto de pruebas que nunca ha visto, visualizando gráficamente dónde acierta y dónde falla.

In [ ]:
@torch.no_grad()
def visualize_predictions(model, dataset, device,
                           num_samples=6, score_threshold=0.5,
                           save_path="detecciones.png"):
    """
    Muestra num_samples imágenes con cajas GT (verde) y predicciones (rojo).
    """
    model.eval()
    indices = np.random.choice(len(dataset), min(num_samples, len(dataset)), replace=False)

    cols = 3
    rows = (len(indices) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes.flatten()

    for ax, idx in zip(axes, indices):
        img_tensor, target = dataset[idx]
        img_np = img_tensor.permute(1, 2, 0).numpy()

        pred = model([img_tensor.to(device)])[0]

        ax.imshow(img_np)
        ax.set_title(f"Muestra {idx}", fontsize=9)
        ax.axis("off")

        # Ground truth (verde)
        for box in target["boxes"]:
            x1, y1, x2, y2 = box.tolist()
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                      linewidth=2, edgecolor="limegreen",
                                      facecolor="none")
            ax.add_patch(rect)

        # Predicciones (rojo)
        mask = (pred["scores"] >= score_threshold) & (pred["labels"] == 1)
        for box, score in zip(pred["boxes"][mask], pred["scores"][mask]):
            x1, y1, x2, y2 = box.tolist()
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                      linewidth=2, edgecolor="tomato",
                                      facecolor="none")
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, f"{score:.2f}", color="tomato",
                    fontsize=7, fontweight="bold")

    # Oculta ejes sobrantes
    for ax in axes[len(indices):]:
        ax.axis("off")

    # Leyenda manual
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color="limegreen", lw=2, label="Ground Truth"),
        Line2D([0], [0], color="tomato",    lw=2, label="Predicción"),
    ]
    fig.legend(handles=legend_elements, loc="lower center", ncol=2,
               fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"Visualización guardada en {save_path}")
    plt.show()

In [ ]:
visualize_predictions(model, val_dataset, device, num_samples=6)